In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import immunowave as iw
from immunowave_paper_utils import style_axes, colors, fontsize, linewidth, rc_params
import pickle
from scipy.stats import linregress
from functools import partial

In [12]:
linewidth = 3
fontsize = 24
markersize = 12
markeredgewidth = 2
rc_params["axes.linewidth"] = linewidth
rc_params["font.size"] = fontsize
mpl.rcParams.update(rc_params)
mpl.rcParams["pdf.fonttype"] = 42
d1_color = colors['wave']
d2_color = np.array([94, 45, 144]) / 255
d3_color = np.array([210, 85, 39]) / 255

color_list = [d1_color, d2_color, d3_color]

In [13]:
savedir = r'/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep'

In [14]:
class State(iw.State):
    u: iw.ScalarField

In [15]:
class Model(iw.Model):
    KD: float
    n: float
    I: float
    k: float = 1.0
    γ: float = 1.0
    D: float = 1.0
    # Bandwidth adjustment for Dirac delta approximation (grid units)
    bw_adjust: float = 0.5

    def f(self, u):
        KD, n, k, γ = self.KD, self.n, self.k, self.γ
        return k * u.hill(KD, n) - γ * u

    def delta(self, u):
        """Helper function to approximate a Dirac delta as a Gaussian on the same grid as u."""
        σ = self.bw_adjust * u.h

        def gaussian(*coords):
            squared_dist = sum(coord**2 for coord in coords)
            return jnp.exp(-squared_dist / (2 * σ**2))

        result = iw.ScalarField(u.values.shape, u.lb, u.h, fn=gaussian)
        result /= result.integral()
        return result

    def __call__(self, t, state: State, args=None):
        u = state.u
        Δu = u.laplacian(bc="neumann")
        f = self.f
        I = self.I
        δ = self.delta(u)
        D = self.D

        dudt = D * Δu + f(u) + 2 * I * δ

        return State(dudt)
    
class Model_mixed(iw.Model):
    KD: float
    n: float
    I: float
    k: float = 1.0
    γ: float = 1.0
    # Bandwidth adjustment for Dirac delta approximation (grid units)
    bw_adjust: float = 0.5

    def f(self, u):
        KD, n, k, γ = self.KD, self.n, self.k, self.γ
        return k * u.hill(KD, n) - γ * u

    def __call__(self, t, state: State, args=None):
        u = state.u
        f = self.f
        I = self.I

        dudt = f(u) + 2 * I

        return State(dudt)

In [47]:
"""load the data"""
with open(savedir + '/1D/D_sweep.pkl', 'rb') as file:
    d1_dict = pickle.load(file)

with open(savedir + '/2D/D_sweep.pkl', 'rb') as file:
    d2_dict = pickle.load(file)
    
with open(savedir + '/3D/D_sweep.pkl', 'rb') as file:
    d3_dict = pickle.load(file)
    
wave_dicts = [d1_dict, d2_dict, d3_dict]

with open(savedir + '/mixed/D_sweep.pkl', 'rb') as file:
    mixed_dict = pickle.load(file)

In [17]:
def fit_intercept_given_slope(x, y, slope):
    return (np.sum(y) - slope * np.sum(x)) / len(x)


In [18]:
%matplotlib qt

## Plot $I_c^{wave}$ vs $D$

In [54]:
d2_error_bars.shape

(2, 6)

In [52]:
# error bars from the optimization log. d=1 had such a fine tolerance the errorbars are too small to see on the plot
# d2_error_bars = np.array([[7.807e-01, 1.024e+00],
#                         [1.961e+00, 2.572e+00],
#                         [4.926e+00, 6.460e+00],
#                         [1.237e+01, 1.623e+01],
#                         [3.108e+01, 4.076e+01],
#                         [7.807e+01, 1.024e+02]]).T

d1_error_bars = []

d2_error_bars = np.array([[1.014e+00, 1.023e+00],
                        [2.394e+00, 2.408e+00],
                        [5.835e+00, 5.871e+00],
                        [1.439e+01, 1.448e+01],
                        [3.592e+01, 3.614e+01],
                        [8.966e+01, 9.022e+01]]).T


d3_error_bars = np.array([[1.522e+01, 2.899e+01],
                        [6.060e+01, 1.154e+02],
                        [1.264e+02, 2.298e+02],
                        [5.031e+02, 9.147e+02],
                        [2.003e+03, 3.641e+03],
                        [7.973e+03, 1.450e+04]]).T


all_error_bars = [d1_error_bars, d2_error_bars, d3_error_bars]

In [57]:
D_grid = d1_dict['D_grid']
plt.figure()
for d in range(3):
    Ic_wave = wave_dicts[d]['Ic_grid']
    if d == 0:
        plt.plot(D_grid / 60, Ic_wave, 'o', color=color_list[d], markerfacecolor='none', markersize=markersize, markeredgewidth=markeredgewidth)
    else:
        plt.errorbar(D_grid / 60, Ic_wave, np.abs(Ic_wave - all_error_bars[d]), 
                     marker='o', 
                     linestyle='none', 
                     color=color_list[d], 
                     markerfacecolor='none', 
                     markersize=markersize, 
                     markeredgewidth=markeredgewidth,
                     capsize=linewidth)

    intercept = fit_intercept_given_slope(np.log10(D_grid), np.log10(Ic_wave), (d + 1) / 2)
    prefactor = 10 ** intercept

    plt.plot(D_grid / 60, prefactor * D_grid ** ((d + 1) / 2), color = color_list[d], linewidth=linewidth)

    
plt.xscale("log")
plt.yscale("log")
plt.xlabel('diffusion coefficient, $D$ ($\mu m^2$/s)', fontsize=fontsize)
plt.ylabel('$I^{wave}_c / k$', fontsize=fontsize)
ax = style_axes(plt.gca(), fontsize=fontsize)   
plt.tight_layout() 
    
    

<>:25: SyntaxWarning: invalid escape sequence '\m'
<>:25: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_114077/138549987.py:25: SyntaxWarning: invalid escape sequence '\m'
  plt.xlabel('diffusion coefficient, $D$ ($\mu m^2$/s)', fontsize=fontsize)


In [58]:
plt.savefig(r'/home/brandon/Documents/Code/immunowave/plots/2026_01_10_Ic_wave_vs_D.pdf')

## Plot $I_c^{wave}/I_c^{mixed}$ vs $D$
(This one is not actually informative)

In [45]:
D_grid = d1_dict['D_grid']
Ic_mixed = mixed_dict['Ic_grid'][0]
plt.figure()
for d in range(3):
    Ic_wave = wave_dicts[d]['Ic_grid']
    if d == 0:
        plt.plot(D_grid / 60, Ic_wave / Ic_mixed, 'o', color=color_list[d], markerfacecolor='none', markersize=markersize, markeredgewidth=markeredgewidth)
    else:
        plt.errorbar(D_grid / 60, Ic_wave / Ic_mixed, all_error_bars[d] / Ic_mixed, marker='o', linestyle='none', color=color_list[d], markerfacecolor='none', markersize=markersize, markeredgewidth=markeredgewidth)

    intercept = fit_intercept_given_slope(np.log10(D_grid), np.log10(Ic_wave / Ic_mixed), (d + 1) / 2)
    prefactor = 10 ** intercept

    plt.plot(D_grid / 60, prefactor * D_grid ** ((d + 1) / 2), color = color_list[d], linewidth=linewidth)

    
plt.xscale("log")
plt.yscale("log")
plt.xlabel('diffusion coefficient, $D$ ($\mu m^2$/s)', fontsize=fontsize)
plt.ylabel('$I^{wave}_c / I^{mixed}_c$ (1/min)', fontsize=fontsize)
ax = style_axes(plt.gca(), fontsize=fontsize)   
plt.tight_layout() 
    

<>:19: SyntaxWarning: invalid escape sequence '\m'
<>:19: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_114077/2167828818.py:19: SyntaxWarning: invalid escape sequence '\m'
  plt.xlabel('diffusion coefficient, $D$ ($\mu m^2$/s)', fontsize=fontsize)
